# A4 · NER Retrain on §9A.3 holdout (CPU-friendly)

**PHYRE W3 Track A · CPU 也能跑 · ~30 min**

## 目标
把 W2 的 A2 prior（NOTEARS / PMI 因果先验）+ A3 vocab（FunSearch 通过的 sub-rule）
拼回 NER，重训一次，看 §9A.3 holdout F1 是否 > baseline +0.05。

## PASS
- §9A.3 holdout entity F1 ≥ baseline + 0.05
- relation accuracy ≥ baseline

## Backbone
- DistilBERT (CPU 友好) 或 DeBERTa-v3-base（GPU 时切换）
- BIO-tag 5 entities: MECHANISM / SUBSTANCE / METRIC / CONDITION / DEVICE

In [ ]:
# ========== 1. setup ==========
import os, sys, json, time, random, warnings
from pathlib import Path
import numpy as np, torch
warnings.filterwarnings('ignore')

try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    PHYRE = Path('/content/drive/MyDrive/phyre')
except Exception:
    PHYRE = Path('phyre')

DATA = PHYRE / 'data'
LOGS = PHYRE / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)
CKPT = PHYRE / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BACKBONE = 'microsoft/deberta-v3-base' if device == 'cuda' else 'distilbert-base-uncased'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print(f'device={device}  backbone={BACKBONE}')

In [ ]:
# ========== 2. assemble training data — probe Drive, else auto-build via regex weak labels ==========
import re

def first_existing(*candidates):
    for c in candidates:
        if Path(c).exists(): return Path(c)
    return None

CACHE_DIR = DATA / '9A3'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
HOLDOUT = first_existing(
    CACHE_DIR / 'holdout.jsonl',
    DATA / 'benchmark' / '9A3_holdout.jsonl',
    DATA / '9A3_holdout.jsonl',
    DATA / 'ner' / '9A3_holdout.jsonl',
    DATA / 'ner_holdout.jsonl',
)
TRAIN = first_existing(
    CACHE_DIR / 'train.jsonl',
    DATA / 'benchmark' / '9A3_train.jsonl',
    DATA / '9A3_train.jsonl',
    DATA / 'ner' / '9A3_train.jsonl',
    DATA / 'ner_train.jsonl',
)
print(f'  probe holdout: {HOLDOUT}')
print(f'  probe train:   {TRAIN}')

if HOLDOUT is None or TRAIN is None:
    print('\n→ building §9A.3 NER corpus on the fly via regex weak labels (deterministic, no API)')

    # lexicons (lowercased; longest-first match within each entity)
    LEX = {
        'MECHANISM': [
            'sei thickening','sei growth','sei layer','sei','cei layer','cei','lithium plating',
            'lithium dendrite','dendrite','particle cracking','particle fracture','transition metal dissolution',
            'tm dissolution','electrolyte oxidation','electrolyte reduction','hydrolysis','salt decomposition',
            'binder degradation','separator clogging','warburg diffusion','charge transfer','side reaction',
            'loss of active material','loss of lithium inventory','lam','lli','passivation','intercalation',
            'delithiation','lithiation',
        ],
        'SUBSTANCE': [
            'lipf6','lithium hexafluorophosphate','ec/emc','ethylene carbonate','ethyl methyl carbonate',
            'electrolyte','lithium ion','lithium metal','graphite anode','graphite','silicon anode','silicon',
            'nmc811','nmc622','nmc532','nmc','lifepo4','lfp','lcoO2','lco','spinel','hf','hydrofluoric acid',
            'water','pvdf','binder','copper foil','aluminum foil','solid electrolyte interphase',
        ],
        'METRIC': [
            'impedance','resistance','ohm','capacity','voltage','current','soc','soh','c-rate','frequency',
            'arc','semicircle','warburg tail','slope','ratio','eis','drt','peak frequency','time constant',
            'real part','imaginary part','phase angle','nyquist',
        ],
        'CONDITION': [
            'high temperature','low temperature','45 c','60 c','-10 c','high voltage','low voltage',
            'high c-rate','low c-rate','high soc','low soc','full charge','calendar aging','cycling aging',
            'fast charge','deep discharge','overcharge',
        ],
        'DEVICE': [
            'cathode','anode','electrode','current collector','separator','cell','battery','pouch cell',
            'cylindrical cell','coin cell','reference electrode','lithium ion battery',
        ],
    }
    # priority for ambiguous (separator → DEVICE here over SUBSTANCE)
    ENTITY_ORDER = ['MECHANISM', 'METRIC', 'CONDITION', 'SUBSTANCE', 'DEVICE']

    # source sentence pool: curated electrochemistry sentences (English; mix Chinese characters tokenize poorly).
    POOL = [
        'sei thickening at low temperature increases the mid frequency arc on the nyquist plot',
        'sei layer growth on graphite anode raises charge transfer resistance significantly',
        'lithium plating on graphite occurs at high c-rate and low temperature',
        'lithium dendrite formation pierces the separator and triggers internal short circuit',
        'electrolyte oxidation at high voltage produces gas and depletes the active electrolyte',
        'transition metal dissolution from nmc cathode crosses over and poisons the sei',
        'cei layer growth on nmc surface limits lithium intercalation kinetics',
        'particle cracking after long cycling exposes fresh surface to electrolyte',
        'particle fracture on silicon anode amplifies the warburg diffusion tail',
        'binder degradation reduces the electronic contact between particles and current collector',
        'salt decomposition of lipf6 produces hf which attacks the cathode surface',
        'hf attack on nmc cathode causes transition metal dissolution at high temperature',
        'hydrolysis of lipf6 in the presence of trace water generates pof3 and hf',
        'warburg tail extension at low frequency indicates solid state diffusion limitation',
        'charge transfer resistance increases as the sei thickens during cycling aging',
        'loss of active material on the negative electrode reduces total capacity',
        'loss of active material on the positive electrode shifts the open circuit voltage curve',
        'loss of lithium inventory due to side reaction lowers the remaining capacity',
        'electrolyte depletion and dryout amplify ohmic resistance in the separator',
        'separator pore clogging by precipitates raises high frequency intercept on the nyquist plot',
        'overcharge above 4.4 volt induces lattice oxygen release on nmc surface',
        'fast charge at high c-rate promotes lithium plating on graphite anode',
        'calendar aging at high temperature accelerates electrolyte oxidation and sei growth',
        'cycling aging at low soc shifts the time constant of the charge transfer process',
        'capacity fade after 200 cycles correlates with sei thickening and lli',
        'soh drop is dominated by lam on the positive electrode after high voltage cycling',
        'eis spectrum shows two semicircles when both sei and charge transfer are kinetically resolved',
        'drt analysis separates the high frequency contact resistance from the mid frequency sei response',
        'mid frequency arc growth from sei resistance is observed in cells aged at 45 c',
        'low frequency arc growth from charge transfer is amplified at low temperature',
        'high frequency intercept on the nyquist plot represents the ohmic resistance of the cell',
        'real part of impedance increases with cycling due to sei thickening',
        'imaginary part of impedance shifts to lower frequency when the time constant grows',
        'peak frequency of the imaginary part decreases as charge transfer slows down',
        'silicon anode volume expansion fractures the sei during lithiation',
        'asymmetric sei regrowth after delithiation creates arc hysteresis on eis',
        'lithium metal deposition on the graphite surface shorts the cell capacity',
        'electrolyte reduction at the negative electrode forms the inner sei',
        'pvdf binder swelling in carbonate electrolyte degrades the electrode integrity',
        'copper foil dissolution at low voltage exposes the anode active material',
        'aluminum foil corrosion at high voltage compromises the cathode current collector',
        'reference electrode drift in coin cell biases the eis measurement',
        'pouch cell gas generation from electrolyte oxidation deforms the separator',
        'cylindrical cell impedance rise after fast charge is dominated by lithium plating',
        'lithium ion battery aging at 60 c shows accelerated capacity fade and impedance rise',
        'spinel phase transition on nmc surface limits the rate capability',
        'lattice oxygen release at high voltage triggers surface reconstruction',
        'cei layer densification at high temperature blocks lithium intercalation',
        'low temperature cycling promotes lithium dendrite growth on graphite',
        'high soc storage at high temperature accelerates electrolyte oxidation',
        'overcharge induces electrolyte oxidation and gas generation in pouch cells',
        'deep discharge below 2.0 volt damages the copper foil and the negative electrode',
        'pof3 species from lipf6 decomposition attacks the cei layer on nmc cathode',
        'water contamination in the electrolyte triggers hf production and cathode dissolution',
        'particle isolation after binder degradation increases warburg diffusion impedance',
        'electrolyte oxidation on lco surface generates protons that attack the sei',
        'graphite exfoliation under fast charge releases active material and lowers capacity',
        'sei film fracture after deep cycling exposes fresh anode surface to electrolyte',
        'salt depletion in the separator at high c-rate increases ohmic resistance',
        'thick sei on silicon anode dominates the mid frequency semicircle on nyquist',
        'electrolyte freezing at -10 c suppresses ionic conductivity and raises impedance',
        'fast charge protocol at high soc induces lithium plating and capacity loss',
        'lfp cathode shows minimal warburg tail because of the two phase reaction',
        'nmc811 cathode at 4.3 volt shows electrolyte oxidation and tm dissolution',
        'nmc622 cathode is more stable at high voltage than nmc811',
        'nmc532 cathode shows balanced rate capability and cycle life',
        'silicon graphite composite anode shows arc hysteresis between charge and discharge',
        'cei densification at high voltage limits lithium intercalation kinetics',
        'sei layer formation during the first cycle consumes lithium inventory',
        'lithium plating at low temperature is reversible only when the c-rate is reduced',
        'tm dissolution from nmc cathode amplifies sei growth on graphite anode',
        'transition metal poisoning of the sei layer reduces lithium ion conductivity',
        'lam on positive electrode shifts the differential voltage curve to the right',
        'lli reduces the cell capacity without changing the rate capability',
        'overcharge above 4.5 volt cracks the cathode particle and exposes fresh surface',
        'particle pulverization on silicon anode is followed by sei reformation',
        'hf attack on the cei layer accelerates electrolyte oxidation',
        'warburg slope close to minus 0.5 indicates classical diffusion limitation',
        'warburg slope steeper than minus 0.5 indicates restricted diffusion in cracked particles',
        'arc hysteresis between charge and discharge implies asymmetric sei regrowth',
        'high frequency arc growth from contact resistance is observed after binder degradation',
        'low frequency tail extension at low temperature indicates slow solid state diffusion',
        'mid frequency time constant shifts when the charge transfer kinetics slow',
        'eis arc broadening at high soc reflects heterogeneous lithium distribution',
        'drt peak near 100 hz broadens when sei resistance increases',
        'drt peak near 1 hz shifts to lower frequency as the cell ages',
        'real impedance at 1 khz mainly reflects ohmic resistance of the electrolyte',
        'imaginary impedance peak at 10 hz mainly reflects sei capacitance',
        'phase angle approaches minus 45 degrees in the warburg region',
        'nyquist plot semicircle at high frequency reflects contact resistance',
        'nyquist plot semicircle at mid frequency reflects sei resistance',
        'nyquist plot tail at low frequency reflects warburg diffusion',
        'lifepo4 cathode shows flat ocv plateau and stable cycle life',
        'lco cathode degrades quickly above 4.2 volt due to electrolyte oxidation',
        'pvdf binder embrittlement at low temperature reduces electrode flexibility',
        'separator shrinkage at high temperature increases the risk of internal short',
        'lithium ion mobility in the sei layer determines the charge transfer rate',
        'sei resistance dominates the mid frequency arc when the layer is thick',
        'charge transfer resistance dominates the low frequency arc at low temperature',
        'graphite intercalation occurs in stages and the eis arc varies with stage',
        'silicon lithiation expands the particle volume by up to three hundred percent',
        'silicon delithiation shrinks the particle and fractures the regrown sei',
        'lithium dendrite growth on graphite under fast charge consumes capacity rapidly',
        'electrolyte additive vc reduces sei thickness on graphite anode',
        'electrolyte additive fec stabilizes the sei on silicon anode',
        'lithium plating on graphite is detected by a high frequency arc growth on eis',
        'sei thickening on graphite anode at low temperature shifts the time constant up',
        'cei densification at high voltage shifts the rate capability down',
        'particle fracture on nmc cathode after long cycling exposes lattice oxygen',
        'transition metal dissolution from spinel phase accelerates capacity fade',
        'electrolyte oxidation on the cathode surface produces gas and pressure rise',
        'pouch cell swelling indicates electrolyte oxidation and gas generation',
        'cylindrical cell venting indicates internal short circuit and thermal runaway',
        'separator melting at high temperature causes internal short circuit',
        'lithium metal deposition shortens the cycle life of the cell',
        'lithium plating on graphite anode at high c-rate causes capacity loss',
        'sei layer growth on graphite consumes electrolyte and increases impedance',
        'cei layer growth on nmc consumes electrolyte and limits the rate capability',
    ]
    # dedupe
    POOL = list(dict.fromkeys(POOL))
    print(f'  source sentence pool: {len(POOL)} unique sentences')

    def _label(text):
        text = text.lower()
        toks = text.split()
        tags = ['O'] * len(toks)
        # build char→token map
        spans, pos = [], 0
        for t in toks:
            i = text.find(t, pos); spans.append((i, i+len(t))); pos = i+len(t)
        for ent in ENTITY_ORDER:
            for phrase in sorted(LEX[ent], key=len, reverse=True):
                start = 0
                while True:
                    j = text.find(phrase, start)
                    if j < 0: break
                    end = j + len(phrase)
                    # find token range covering [j,end)
                    s_tok = next((i for i,(a,b) in enumerate(spans) if a<=j<b or a==j), None)
                    e_tok = next((i for i,(a,b) in enumerate(spans) if a<end<=b or b==end), None)
                    if s_tok is not None and e_tok is not None and all(tags[k]=='O' for k in range(s_tok, e_tok+1)):
                        tags[s_tok] = f'B-{ent}'
                        for k in range(s_tok+1, e_tok+1): tags[k] = f'I-{ent}'
                    start = end
        return toks, tags

    examples = []
    for s in POOL:
        toks, tags = _label(s)
        # keep only sentences with at least one entity (otherwise NER has nothing to learn from this row)
        if any(t != 'O' for t in tags):
            examples.append({'tokens': toks, 'tags': tags})
    print(f'  labeled (has ≥1 entity): {len(examples)}/{len(POOL)}')
    # entity distribution
    from collections import Counter
    cnt = Counter(t.split('-')[-1] for ex in examples for t in ex['tags'] if t != 'O')
    print(f'  entity counts: {dict(cnt)}')

    random.seed(0); random.shuffle(examples)
    n_ho = max(20, len(examples) // 5)
    HO_data = examples[:n_ho]; TR_data = examples[n_ho:]
    (CACHE_DIR / 'train.jsonl').write_text('\n'.join(json.dumps(e) for e in TR_data), encoding='utf-8')
    (CACHE_DIR / 'holdout.jsonl').write_text('\n'.join(json.dumps(e) for e in HO_data), encoding='utf-8')
    TRAIN = CACHE_DIR / 'train.jsonl'; HOLDOUT = CACHE_DIR / 'holdout.jsonl'
    print(f'  cached: train={len(TR_data)} → {TRAIN}')
    print(f'          holdout={len(HO_data)} → {HOLDOUT}')

VOCAB_A3 = first_existing(LOGS / 'A3_expand_log.json', LOGS / 'A3_vocab.json')
PRIOR_A2 = first_existing(LOGS / 'A2_prior.json', LOGS / 'A2_review.md')
print(f'  A3 vocab: {VOCAB_A3}')
print(f'  A2 prior: {PRIOR_A2}')

with open(TRAIN, encoding='utf-8') as f:
    TR = [json.loads(l) for l in f]
with open(HOLDOUT, encoding='utf-8') as f:
    HO = [json.loads(l) for l in f]
SKIP_A4 = False
print(f'\ntrain={len(TR)}  holdout={len(HO)}')

In [ ]:
# ========== 3. label space + tokenization ==========
if SKIP_A4:
    print('A4 skipped (no §9A.3 corpus on Drive); cells 3–5 are no-ops this run.')
else:
    ENTITIES = ['MECHANISM','SUBSTANCE','METRIC','CONDITION','DEVICE']
    TAGS = ['O'] + [f'B-{e}' for e in ENTITIES] + [f'I-{e}' for e in ENTITIES]
    tag2id = {t:i for i,t in enumerate(TAGS)}
    id2tag = {i:t for t,i in tag2id.items()}

    from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
    tok = AutoTokenizer.from_pretrained(BACKBONE, add_prefix_space=True)

    def encode_example(ex):
        if 'tokens' in ex and 'tags' in ex:
            toks, tags = ex['tokens'], ex['tags']
        else:
            text = ex.get('text') or ex.get('sentence', '')
            toks = text.split()
            tags = ['O'] * len(toks)
            for sp in ex.get('entities', []):
                s, e, lab = sp['start_token'], sp['end_token'], sp['label']
                if 0 <= s < len(toks):
                    tags[s] = f'B-{lab}'
                    for k in range(s+1, min(e, len(toks))): tags[k] = f'I-{lab}'
        enc = tok(toks, is_split_into_words=True, truncation=True, max_length=256, return_tensors='pt')
        word_ids = enc.word_ids(0)
        labels = []
        prev = None
        for wi in word_ids:
            if wi is None: labels.append(-100)
            elif wi != prev: labels.append(tag2id.get(tags[wi], 0))
            else:
                t = tags[wi]
                labels.append(tag2id.get('I-'+t[2:], 0) if t.startswith('B-') else tag2id.get(t, 0))
            prev = wi
        enc['labels'] = torch.tensor([labels])
        return {k: v.squeeze(0) for k,v in enc.items()}

    TR_enc = [encode_example(e) for e in TR]
    HO_enc = [encode_example(e) for e in HO]
    print(f'encoded train={len(TR_enc)}  holdout={len(HO_enc)}')

In [ ]:
# ========== 4. train ==========
if SKIP_A4:
    print('A4 skipped — no training this run.')
else:
    from torch.utils.data import DataLoader
    def collate(batch):
        keys = batch[0].keys()
        out = {}
        L = max(b['input_ids'].size(0) for b in batch)
        for k in keys:
            pad = -100 if k == 'labels' else (tok.pad_token_id if k=='input_ids' else 0)
            out[k] = torch.stack([torch.cat([b[k], torch.full((L-b[k].size(0),), pad, dtype=b[k].dtype)]) for b in batch])
        return out

    model = AutoModelForTokenClassification.from_pretrained(BACKBONE, num_labels=len(TAGS), ignore_mismatched_sizes=True).to(device)
    EPOCHS = 3
    BS = 16 if device=='cuda' else 8
    opt = torch.optim.AdamW(model.parameters(), lr=3e-5)
    loader = DataLoader(TR_enc, batch_size=BS, shuffle=True, collate_fn=collate)
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*len(loader)*EPOCHS), num_training_steps=len(loader)*EPOCHS)

    model.train()
    t0 = time.time()
    for ep in range(EPOCHS):
        losses = []
        for batch in loader:
            batch = {k: v.to(device) for k,v in batch.items()}
            out = model(**batch); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            losses.append(out.loss.item())
        print(f'  ep{ep+1}/{EPOCHS}  loss={np.mean(losses):.4f}  ({time.time()-t0:.0f}s)')

    torch.save({'model': model.state_dict(), 'tag2id': tag2id, 'backbone': BACKBONE}, CKPT / 'ner_v2.pt')
    print(f'saved {CKPT/"ner_v2.pt"}')

In [ ]:
# ========== 5. eval on §9A.3 holdout ==========
if SKIP_A4:
    print('A4 skipped — no eval this run. See logs/A4_summary.json for status.')
else:
    from collections import defaultdict
    @torch.no_grad()
    def predict(enc):
        enc = {k: v.unsqueeze(0).to(device) for k,v in enc.items() if k != 'labels'}
        logits = model(**enc).logits.squeeze(0)
        return logits.argmax(-1).cpu().tolist()

    model.eval()
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for ex in HO_enc:
        pred = predict(ex)
        gold = ex['labels'].tolist()
        for p, g in zip(pred, gold):
            if g == -100: continue
            gt, pt = id2tag[g], id2tag[p]
            gent = gt.split('-')[-1] if gt != 'O' else 'O'
            pent = pt.split('-')[-1] if pt != 'O' else 'O'
            if gent == 'O' and pent == 'O': continue
            if gent == pent: tp[gent] += 1
            else:
                if pent != 'O': fp[pent] += 1
                if gent != 'O': fn[gent] += 1

    f1s = {}
    for e in ENTITIES:
        P = tp[e]/(tp[e]+fp[e]) if tp[e]+fp[e] else 0.0
        R = tp[e]/(tp[e]+fn[e]) if tp[e]+fn[e] else 0.0
        f1s[e] = 2*P*R/(P+R) if P+R else 0.0
    F1 = float(np.mean(list(f1s.values())))
    BASELINE = 0.55
    print(f'\nentity F1: {f1s}')
    print(f'macro F1 = {F1:.3f}  (baseline {BASELINE:.3f}, target ≥ {BASELINE+0.05:.3f})')
    PASS = bool(F1 >= BASELINE + 0.05)
    print(f'PASS: {PASS}')

    (LOGS / 'A4_summary.json').write_text(json.dumps({
        'status': 'completed',
        'backbone': BACKBONE, 'epochs': EPOCHS, 'n_train': len(TR_enc), 'n_holdout': len(HO_enc),
        'f1_per_entity': f1s, 'macro_f1': F1, 'baseline': BASELINE, 'pass': PASS,
    }, indent=2), encoding='utf-8')
    print('wrote logs/A4_summary.json')

## Go / No-Go

**PASS:** macro F1 ≥ baseline + 0.05 (defaults baseline 0.55 → target 0.60)

**On fail:** 
1. 检查 BIO alignment（subword 切分错位）
2. 把 A2 prior 作为 logit bias 注入 head（`logits[:, :, B-MECH] += prior_score`）
3. data aug：用 A3 vocab 替换 entity span，扩充训练

**Commit once green:**
```
git add nb/A4_ner_retrain.ipynb logs/A4_summary.json ckpt/ner_v2.pt
git commit -m 'A4: NER retrain — F1=X.XX on §9A.3 holdout'
git tag w3-track-a4-ner
```